# CELL 01

# Knowledge Graph — Notebook 5
## Knowledge Graph vs Relational Database vs Vector Store

### Running Problem: Travel Planning

In the previous notebooks, we built a Travel Planning
Knowledge Graph.

We have seen that a Knowledge Graph is particularly useful
when information is connected through relationships.

But an important question remains:

> Is a Knowledge Graph always the best way to represent data?

The answer is:

> No.

Different data representations are suitable for
different types of problems.

In this notebook, we will represent the SAME travel
knowledge in three different ways:

1. Relational Database
2. Knowledge Graph
3. Vector Store

Then we will ask similar travel questions and compare
how each representation works.

### Main Learning Goal

Do not ask:

    "Which technology is the best?"

Instead ask:

    "Which representation is appropriate for
     this particular problem?"

# CELL 02

# The Three Representations

We will use the same travel information.

### Representation 1 — Relational Database

Information is organized into tables.

    Tables
       ↓
    Rows + Columns
       ↓
    SQL Queries

### Representation 2 — Knowledge Graph

Information is represented as:

    Entity → Relationship → Entity/Value

       ↓
    Graph Traversal
       ↓
    Multi-Hop Queries

### Representation 3 — Vector Store

Information is represented as vectors.

    Text
      ↓
    Embedding
      ↓
    Vector
      ↓
    Similarity Search

Each representation answers different kinds
of questions effectively.

                 SAME TRAVEL KNOWLEDGE
                          │
          ┌───────────────┼───────────────┐
          ↓               ↓               ↓
      TABLE           KNOWLEDGE         VECTOR
      DATABASE          GRAPH            STORE
          │               │               │
          ↓               ↓               ↓
         SQL          RELATIONSHIPS     SIMILARITY

# CELL 03

# Our Travel Knowledge

Let us use the following information.

- Chennai is located in Tamil Nadu.
- Mahabalipuram is near Chennai.
- Mahabalipuram is a heritage destination.
- Marina Beach is located in Chennai.
- Chennai is connected to Bengaluru.
- Bengaluru is connected to Mysuru.
- Hotel SeaView is located in Mahabalipuram.
- Hotel SeaView costs ₹3500 per night.
- Pondicherry is near Chennai.
- Pondicherry is a heritage destination.
- Hotel Heritage is located in Pondicherry.
- Hotel Heritage costs ₹3000 per night.

The important point is:

> We will NOT change the knowledge.

We will change only the way in which
the knowledge is represented.

In [1]:
# CELL 04

travel_facts = [
    "Chennai is located in Tamil Nadu.",
    "Mahabalipuram is near Chennai.",
    "Mahabalipuram is a heritage destination.",
    "Marina Beach is located in Chennai.",
    "Chennai is connected to Bengaluru.",
    "Bengaluru is connected to Mysuru.",
    "Hotel SeaView is located in Mahabalipuram.",
    "Hotel SeaView costs 3500 rupees per night.",
    "Pondicherry is near Chennai.",
    "Pondicherry is a heritage destination.",
    "Hotel Heritage is located in Pondicherry.",
    "Hotel Heritage costs 3000 rupees per night."
]

for fact in travel_facts:
    print(fact)

Chennai is located in Tamil Nadu.
Mahabalipuram is near Chennai.
Mahabalipuram is a heritage destination.
Marina Beach is located in Chennai.
Chennai is connected to Bengaluru.
Bengaluru is connected to Mysuru.
Hotel SeaView is located in Mahabalipuram.
Hotel SeaView costs 3500 rupees per night.
Pondicherry is near Chennai.
Pondicherry is a heritage destination.
Hotel Heritage is located in Pondicherry.
Hotel Heritage costs 3000 rupees per night.


# CELL 05

# Part 1 — First Representation: Relational Database

A relational database stores information in tables.

For our travel problem, we could create tables such as:

    DESTINATIONS
    HOTELS
    CONNECTIONS

For example:

DESTINATIONS

    id | name          | category
    --------------------------------
    1  | Mahabalipuram | Heritage
    2  | Pondicherry   | Heritage

HOTELS

    id | name           | destination     | price
    ------------------------------------------------
    1  | Hotel SeaView  | Mahabalipuram   | 3500
    2  | Hotel Heritage | Pondicherry     | 3000

CONNECTIONS

    from_city | to_city
    --------------------
    Chennai   | Bengaluru
    Bengaluru | Mysuru

Notice that the relationship information is represented
through columns and rows.

# CELL 06

## Why Use SQLite?

SQLite is a relational database that is:

- lightweight
- local
- serverless
- easy to use
- available through Python's standard library

Therefore, we can demonstrate relational databases
without installing a database server.

We will use:

    sqlite3

This notebook will remain lightweight.

In [2]:
# CELL 07

import sqlite3

print("SQLite is available through Python.")

SQLite is available through Python.


# CELL 08

## Create a Relational Database

We will create a local SQLite database.

For teaching purposes, we will use an in-memory database.

That means:

    Database exists only while this notebook runs.

We will create three tables:

    destinations
    hotels
    connections

In [3]:
# CELL 09

connection = sqlite3.connect(":memory:")

cursor = connection.cursor()

print("SQLite database created.")

SQLite database created.


In [4]:
# CELL 10

cursor.execute("""
CREATE TABLE destinations (
    id INTEGER PRIMARY KEY,
    name TEXT,
    category TEXT
)
""")

cursor.execute("""
CREATE TABLE hotels (
    id INTEGER PRIMARY KEY,
    name TEXT,
    destination TEXT,
    price INTEGER
)
""")

cursor.execute("""
CREATE TABLE connections (
    from_city TEXT,
    to_city TEXT
)
""")

connection.commit()

print("Tables created.")

Tables created.


# CELL 11

## Insert Destination Data

Our destination table will contain:

    Mahabalipuram → Heritage
    Pondicherry   → Heritage

We are now converting the same travel knowledge
into a relational representation.

In [5]:
# CELL 12

destinations = [
    (1, "Mahabalipuram", "Heritage"),
    (2, "Pondicherry", "Heritage")
]

cursor.executemany(
    "INSERT INTO destinations VALUES (?, ?, ?)",
    destinations
)

connection.commit()

print("Destination data inserted.")

Destination data inserted.


In [6]:
# CELL 13

hotels = [
    (1, "Hotel SeaView", "Mahabalipuram", 3500),
    (2, "Hotel Heritage", "Pondicherry", 3000)
]

cursor.executemany(
    "INSERT INTO hotels VALUES (?, ?, ?, ?)",
    hotels
)

connection.commit()

print("Hotel data inserted.")

Hotel data inserted.


In [7]:
# CELL 14

connections_data = [
    ("Chennai", "Bengaluru"),
    ("Bengaluru", "Mysuru")
]

cursor.executemany(
    "INSERT INTO connections VALUES (?, ?)",
    connections_data
)

connection.commit()

print("Connection data inserted.")

Connection data inserted.


# CELL 15

# Querying the Relational Database

Let us ask:

> Which hotels cost less than ₹4000?

This is a very natural relational database question.

We have a column:

    price

Therefore, SQL can directly filter on that column.

In [8]:
# CELL 16

cursor.execute("""
SELECT name, destination, price
FROM hotels
WHERE price < 4000
""")

results = cursor.fetchall()

for row in results:
    print(row)

('Hotel SeaView', 'Mahabalipuram', 3500)
('Hotel Heritage', 'Pondicherry', 3000)


SELECT name, price
FROM hotels
WHERE price < 3500;

# CELL 18

# Part 2 — The Same Question Using a Knowledge Graph

Now let us ask:

> Which hotels cost less than ₹4000?

In our Knowledge Graph, the information is represented as:

    Hotel SeaView
        ↓ PRICE_PER_NIGHT
        3500

    Hotel Heritage
        ↓ PRICE_PER_NIGHT
        3000

We can inspect the PRICE_PER_NIGHT relationships
and apply the condition.

The answer is also straightforward.

This shows an important point:

> The same question may be easy in more than one representation.

In [9]:
# CELL 19

kg_triples = [
    ("Chennai", "LOCATED_IN", "Tamil Nadu"),
    ("Mahabalipuram", "NEAR", "Chennai"),
    ("Mahabalipuram", "HAS_CATEGORY", "Heritage"),
    ("Marina Beach", "LOCATED_IN", "Chennai"),
    ("Chennai", "CONNECTED_TO", "Bengaluru"),
    ("Bengaluru", "CONNECTED_TO", "Mysuru"),
    ("Hotel SeaView", "LOCATED_IN", "Mahabalipuram"),
    ("Hotel SeaView", "PRICE_PER_NIGHT", 3500),
    ("Pondicherry", "NEAR", "Chennai"),
    ("Pondicherry", "HAS_CATEGORY", "Heritage"),
    ("Hotel Heritage", "LOCATED_IN", "Pondicherry"),
    ("Hotel Heritage", "PRICE_PER_NIGHT", 3000)
]

budget = 4000

for subject, relationship, object_ in kg_triples:
    if relationship == "PRICE_PER_NIGHT" and object_ < budget:
        print(subject, "→ ₹", object_)

Hotel SeaView → ₹ 3500
Hotel Heritage → ₹ 3000


# CELL 20

# A More Interesting Question

Now consider:

> Find hotels in heritage destinations near Chennai.

This question involves several relationships.

We need:

    Hotel
      ↓ LOCATED_IN
    Destination
      ↓ NEAR
    Chennai

and:

    Destination
      ↓ HAS_CATEGORY
    Heritage

This is a relationship-oriented question.

This is where Knowledge Graph representation
starts becoming particularly interesting.

In [10]:
# CELL 21

# Find places near Chennai

near_chennai = set()

for subject, relationship, object_ in kg_triples:
    if relationship == "NEAR" and object_ == "Chennai":
        near_chennai.add(subject)

# Keep only heritage destinations

heritage_near_chennai = set()

for subject, relationship, object_ in kg_triples:
    if (
        relationship == "HAS_CATEGORY"
        and object_ == "Heritage"
        and subject in near_chennai
    ):
        heritage_near_chennai.add(subject)

# Find hotels in those destinations

answers = []

for subject, relationship, object_ in kg_triples:
    if (
        relationship == "LOCATED_IN"
        and object_ in heritage_near_chennai
    ):
        answers.append(subject)

print("Hotels:")
for hotel in answers:
    print("-", hotel)

Hotels:
- Hotel SeaView
- Hotel Heritage


# CELL 22

## What Did We Do?

We followed relationships:

    Hotel
      ↓
    Destination
      ↓
    Chennai

and also checked:

    Destination
      ↓
    Heritage

The important feature is that the question
is naturally expressed as a path through relationships.

This is a strength of graph representation.

# CELL 23

# Part 3 — Solve the Same Question with SQL

Now let us return to the relational database.

Question:

> Find hotels in heritage destinations near Chennai.

Our database does not have a direct:

    Hotel → Destination → Chennai

graph path.

Instead, the required information is distributed
across tables.

We need to reconstruct the relationship using SQL.

This usually involves JOIN operations.

This is not a weakness of relational databases.

JOINs are one of the fundamental strengths of SQL.

But for highly connected, multi-hop relationship questions,
the query can become more complex.

In [11]:
# CELL 24

cursor.execute("""
SELECT
    hotels.name,
    hotels.destination,
    hotels.price
FROM hotels
JOIN destinations
    ON hotels.destination = destinations.name
WHERE destinations.category = 'Heritage'
""")

results = cursor.fetchall()

for row in results:
    print(row)

('Hotel SeaView', 'Mahabalipuram', 3500)
('Hotel Heritage', 'Pondicherry', 3000)


# CELL 25

## What About "Near Chennai"?

Our current relational schema does not directly
store:

    destination → near → Chennai

We would need another table.

For example:

    nearby_places

with:

    place
    nearby_city

This illustrates an important database design principle:

> The way we represent relationships affects
> the way we query them.

In [12]:
# CELL 26

cursor.execute("""
CREATE TABLE nearby_places (
    place TEXT,
    nearby_city TEXT
)
""")

nearby_data = [
    ("Mahabalipuram", "Chennai"),
    ("Pondicherry", "Chennai")
]

cursor.executemany(
    "INSERT INTO nearby_places VALUES (?, ?)",
    nearby_data
)

connection.commit()

print("Nearby-place data inserted.")

Nearby-place data inserted.


# CELL 27

## Now the SQL Query Requires More Relationships

We want:

    Hotel
       ↓
    Destination
       ↓
    Near Chennai

In relational form, we need:

    hotels
       ↓ JOIN
    destinations
       ↓ JOIN
    nearby_places

This is still completely possible.

The point is NOT:

    "SQL cannot do this."

The correct point is:

> Relational databases represent relationships through
> tables, keys and joins.

Knowledge Graphs represent relationships directly
as graph edges.

In [13]:
# CELL 28

cursor.execute("""
SELECT
    hotels.name,
    hotels.destination,
    hotels.price
FROM hotels
JOIN destinations
    ON hotels.destination = destinations.name
JOIN nearby_places
    ON destinations.name = nearby_places.place
WHERE destinations.category = 'Heritage'
  AND nearby_places.nearby_city = 'Chennai'
  AND hotels.price < 4000
""")

results = cursor.fetchall()

for row in results:
    print(row)

('Hotel SeaView', 'Mahabalipuram', 3500)
('Hotel Heritage', 'Pondicherry', 3000)


# CELL 29

# Important Observation

Both systems can answer the question.

Relational Database:

    Tables
      ↓
    JOIN
      ↓
    JOIN
      ↓
    Filter
      ↓
    Answer

Knowledge Graph:

    Hotel
      ↓
    LOCATED_IN
      ↓
    Destination
      ↓
    NEAR
      ↓
    Chennai

The question is not:

    "Can SQL do it?"

It can.

The more useful question is:

> Which representation makes the relationship
> structure more natural for the problem?

# CELL 30

# Part 4 — Introduce the Vector Store

Now we introduce a third representation.

Suppose we store travel descriptions such as:

    "Mahabalipuram is a heritage destination near Chennai."

    "Pondicherry is a heritage destination near Chennai."

    "Hotel SeaView is located in Mahabalipuram
     and costs ₹3500 per night."

Instead of representing each fact explicitly as a triple,
we can represent the text using vectors.

The process is:

    Text
      ↓
    Embedding Model
      ↓
    Vector
      ↓
    Vector Store

A vector captures semantic information about the text.

# CELL 31

## What is an Embedding?

An embedding converts text into a numerical vector.

For example:

    "Mahabalipuram is a heritage destination."

might become conceptually:

    [0.21, -0.14, 0.72, 0.08, ...]

The actual vector will contain many dimensions.

The important idea is:

> Similar meanings tend to produce vectors
> that are close to each other in vector space.

This allows us to perform semantic similarity search.

# CELL 32

## We Will Start with a Simple Demonstration

To understand the idea without introducing
another embedding library immediately, we will first
represent a few travel descriptions using
simple manually constructed vectors.

This is NOT a real embedding model.

It is only a teaching device.

Later, we can replace these vectors with
real embeddings generated by an embedding model
such as an Ollama embedding model.

In [14]:
# CELL 33

documents = [
    "Mahabalipuram is a heritage destination near Chennai.",
    "Pondicherry is a heritage destination near Chennai.",
    "Hotel SeaView is located in Mahabalipuram and costs 3500 rupees.",
    "Hotel Heritage is located in Pondicherry and costs 3000 rupees.",
    "Bengaluru is connected to Mysuru."
]

print("Number of documents:", len(documents))

for document in documents:
    print("-", document)

Number of documents: 5
- Mahabalipuram is a heritage destination near Chennai.
- Pondicherry is a heritage destination near Chennai.
- Hotel SeaView is located in Mahabalipuram and costs 3500 rupees.
- Hotel Heritage is located in Pondicherry and costs 3000 rupees.
- Bengaluru is connected to Mysuru.


# CELL 34

## A Very Important Distinction

A Knowledge Graph explicitly stores:

    Mahabalipuram
        ↓ NEAR
    Chennai

A vector representation does NOT normally store
the relationship as an explicit labelled edge.

Instead, the meaning of the text is encoded
in numerical form.

Therefore:

Knowledge Graph:

    explicit relationships

Vector Store:

    semantic similarity

# CELL 35

# Similarity Search

Suppose the user asks:

> I want a historical place close to Chennai.

This is not an exact keyword match.

The query contains concepts such as:

    historical
    place
    close
    Chennai

A vector store can search for documents
that are semantically similar to this query.

This is one of the major strengths of vector search.

# CELL 36

## Important Limitation of Our Demonstration

We are not going to pretend that manually invented
vectors are real embeddings.

A real vector store requires:

    Embedding Model
          ↓
    Text → Vector
          ↓
    Similarity Calculation
          ↓
    Top-K Results

Our current demonstration is only to understand
the representation concept.

In a later RAG notebook, we can use a real
embedding model and a real vector store.

# CELL 36A

# From Demonstration to a Real Vector Store

So far, we used manually constructed vectors only
to understand the idea of similarity.

Now we will implement a REAL vector retrieval system.

We will use:

    Travel Documents
          ↓
    Ollama Embedding Model
          ↓
    Embedding Vectors
          ↓
    Chroma Vector Store
          ↓
    Similarity Search

This will allow us to experience how a real
vector store works.

### Tools

- Ollama — local embedding model
- Chroma — vector store

Everything will run locally.

# CELL 36C

## Choose an Embedding Model

We need an embedding model to convert text
into numerical vectors.

We will use a local Ollama embedding model.

For example:

    nomic-embed-text

The important distinction is:

    Language Model
        ↓
    Generates text

    Embedding Model
        ↓
    Generates vectors

For vector search, we need the second one.

In [16]:
!pip install -q google-genai chromadb

In [17]:
import os
import getpass
import chromadb
from google import genai


# 1. Set the API key environment variable
os.environ["GEMINI_API_KEY"] = "GEMINI_API_KEY"

# 2. Initialize Gemini Client & Define Model
client = genai.Client()
EMBEDDING_MODEL = "gemini-embedding-001"

print("Gemini Client and ChromaDB initialized successfully.")
print("Embedding model selected:", EMBEDDING_MODEL)

# 3. Create an in-memory ChromaDB collection
chroma_client = chromadb.Client()

# Reset or recreate collection for a clean rerun
if "travel_knowledge" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("travel_knowledge")

collection = chroma_client.create_collection(name="travel_knowledge")




Gemini Client and ChromaDB initialized successfully.
Embedding model selected: gemini-embedding-001


# CELL 36E

## Create Travel Documents for Semantic Search

A vector store normally stores documents or text chunks,
rather than explicit triples.

We will use the same travel domain.

Notice that these are natural-language descriptions.

The vector store will represent their meaning
through embeddings.

In [18]:
# CELL 36F

travel_documents = [
    "Mahabalipuram is a heritage destination near Chennai. "
    "It is known for its historic monuments and peaceful surroundings.",

    "Pondicherry is a heritage destination near Chennai. "
    "It is suitable for a relaxing weekend trip.",

    "Hotel SeaView is located in Mahabalipuram. "
    "The price is 3500 rupees per night.",

    "Hotel Heritage is located in Pondicherry. "
    "The price is 3000 rupees per night.",

    "Bengaluru is connected to Mysuru. "
    "Mysuru is known for its heritage attractions."
]

for i, document in enumerate(travel_documents):
    print(f"{i}: {document}")

0: Mahabalipuram is a heritage destination near Chennai. It is known for its historic monuments and peaceful surroundings.
1: Pondicherry is a heritage destination near Chennai. It is suitable for a relaxing weekend trip.
2: Hotel SeaView is located in Mahabalipuram. The price is 3500 rupees per night.
3: Hotel Heritage is located in Pondicherry. The price is 3000 rupees per night.
4: Bengaluru is connected to Mysuru. Mysuru is known for its heritage attractions.


# CELL 36G

## Convert Travel Documents into Real Embeddings

Now the embedding model converts each document
into a numerical vector.

Conceptually:

    Document
        ↓
    Embedding Model
        ↓
    Vector

Unlike our earlier demonstration,
these vectors are generated by a real embedding model.

In [19]:
# CELL 36H


#  Generate embeddings for your travel documents
embedding_response = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=travel_documents
)

#embeddings = embedding_response["embeddings"]
embeddings = embedding_response.embeddings

print("Number of documents:", len(embeddings))
print("Dimensions of first embedding:", len(embeddings[0].values))

Number of documents: 5
Dimensions of first embedding: 3072


# CELL 36I

## Inspect an Embedding

Let us look at the embedding generated for
the first travel document.

We will not interpret each individual number.

The important point is:

> The embedding is a numerical representation
> of the semantic information in the document.

In [20]:
# CELL 36J

print("First document:")
print(travel_documents[0])

print("\nFirst 10 embedding values:")

# Access .values FIRST, then slice [:10]
print(embeddings[0].values[:10])


print("\nTotal dimensions:")
print(len(embeddings[0].values))

First document:
Mahabalipuram is a heritage destination near Chennai. It is known for its historic monuments and peaceful surroundings.

First 10 embedding values:
[-0.014950898, -0.013487309, 0.00086625037, -0.08201626, -0.011866909, 0.0114873955, -0.001886443, 0.005867875, 0.007637138, 0.009546975]

Total dimensions:
3072


# CELL 36K

## Store the Embeddings in Chroma

Generating embeddings is only one part of
vector retrieval.

We need somewhere to store them.

Chroma will act as our vector store.

The basic pipeline is:

    Documents
        ↓
    Embeddings
        ↓
    Chroma Collection
        ↓
    Similarity Search

In [21]:
# CELL 36L

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="travel_knowledge"
)

print("Chroma collection created.")

Chroma collection created.


In [22]:
# CELL 36M
# Extract the raw float list from each ContentEmbedding object
raw_embeddings = [e.values for e in embeddings]

collection.add(
    ids=[f"travel_{i}" for i in range(len(travel_documents))],
    documents=travel_documents,
    embeddings=raw_embeddings
)

print("Documents and embeddings added to Chroma.")


Documents and embeddings added to Chroma.


# CELL 36N

# Semantic Search

Now we will ask a question using natural language.

Question:

> I want a peaceful historical place near Chennai
> for a weekend trip.

Notice that the question does not exactly repeat
the wording of our stored documents.

This is the situation where semantic retrieval
becomes useful.

In [23]:
# CELL 36O

query = (
    "I want a peaceful historical place near Chennai "
    "for a weekend trip."
)

# query_response = ollama.embed(
#     model=EMBEDDING_MODEL,
#     input=query
# )

query_response = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=query
)
# 2. Access the embedding list using dot notation, then extract .values
query_embedding = query_response.embeddings[0].values

print("Query:")
print(query)

print("\nQuery embedding dimensions:")
print(len(query_embedding))


Query:
I want a peaceful historical place near Chennai for a weekend trip.

Query embedding dimensions:
3072


In [24]:
# CELL 36P

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:")
print(query)

print("\nRetrieved documents:")

for document in results["documents"][0]:
    print("-", document)

Query:
I want a peaceful historical place near Chennai for a weekend trip.

Retrieved documents:
- Pondicherry is a heritage destination near Chennai. It is suitable for a relaxing weekend trip.
- Mahabalipuram is a heritage destination near Chennai. It is known for its historic monuments and peaceful surroundings.
- Hotel Heritage is located in Pondicherry. The price is 3000 rupees per night.


# CELL 36Q

## What Just Happened?

We asked:

    "I want a peaceful historical place near Chennai
     for a weekend trip."

The stored documents did not need to contain
exactly the same wording.

The vector store:

    Question
        ↓
    Query Embedding
        ↓
    Compare with Document Embeddings
        ↓
    Find Similar Vectors
        ↓
    Return Top-K Documents

This is semantic retrieval.

### Key observation

The vector store is answering:

> "Which stored text is most similar in meaning
> to this question?"

It is NOT answering:

> "Which graph path satisfies this relationship?"

# CELL 36R

# A Deliberately Different Question

Now ask:

> Which hotels cost less than 3500 rupees?

This is an exact numerical condition.

Let us see what a vector search returns.

Do not assume the answer beforehand.

Run the experiment and inspect the result.

In [25]:
# CELL 36S

query_2 = "Which hotels cost less than 3500 rupees?"

# 1. Generate query embedding using Gemini API ('contents' parameter)
query_2_response = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=query_2
)

# 2. Extract the raw float list using dot notation and .values
query_2_embedding = query_2_response.embeddings[0].values

# 3. Query ChromaDB with the float vector
results_2 = collection.query(
    query_embeddings=[query_2_embedding],
    n_results=3
)

print("Query:")
print(query_2)

print("\nRetrieved documents:")

for document in results_2["documents"][0]:
    print("-", document)



Query:
Which hotels cost less than 3500 rupees?

Retrieved documents:
- Hotel SeaView is located in Mahabalipuram. The price is 3500 rupees per night.
- Hotel Heritage is located in Pondicherry. The price is 3000 rupees per night.
- Pondicherry is a heritage destination near Chennai. It is suitable for a relaxing weekend trip.


# CELL 36T

## What Did We Learn?

The vector store may retrieve documents mentioning
hotels and prices.

But semantic similarity does NOT guarantee
correct numerical filtering.

The question:

    price < 3500

is an exact structured condition.

A relational database can express this directly:

    WHERE price < 3500

Therefore:

> Vector retrieval is not a replacement for
> exact structured querying.

This is why choosing the representation matters.

# CELL 37

# Part 5 — Compare the Three Representations

Let us summarize the basic difference.

### Relational Database

Represents:

    rows
    columns
    tables
    keys
    relationships through joins

Strong for:

    structured data
    transactions
    filtering
    aggregation
    exact conditions

### Knowledge Graph

Represents:

    entities
    relationships
    properties
    paths

Strong for:

    connected information
    relationship queries
    multi-hop traversal
    explainable paths

### Vector Store

Represents:

    embeddings
    numerical vectors
    semantic similarity

Strong for:

    semantic search
    similarity search
    finding relevant text
    RAG retrieval

# CELL 38

# Part 6 — Same Question, Different Representations

Consider:

> Find hotels under ₹3500.

### Relational Database

Very natural:

    WHERE price < 3500

### Knowledge Graph

Possible through:

    Hotel
      ↓ PRICE_PER_NIGHT
    Price

then apply:

    Price < 3500

### Vector Store

Not the natural choice.

Why?

Because:

    "under ₹3500"

is a structured numerical constraint.

Vector similarity is not designed primarily
for exact numerical filtering.

This is an important lesson:

> Do not use semantic similarity when
> an exact structured query is required.

# CELL 39

# Another Question

Consider:

> Which places near Chennai are connected
> to cities that lead toward Mysuru?

This is a relationship-heavy question.

The reasoning may involve:

    Chennai
       ↓
    connected city
       ↓
    another city
       ↓
    Mysuru

This kind of multi-hop relationship reasoning
is naturally suited to a Knowledge Graph.

A relational database can also answer it,
but may require multiple joins or recursive queries.

A vector store is generally not the natural
representation for this explicit graph traversal.

# CELL 40

# Another Question

Consider:

> I want a peaceful historical place near Chennai
> for a weekend trip.

This is a semantic question.

The user may not use exactly the same words
as the stored descriptions.

For example:

    peaceful
    historical
    weekend trip
    close to Chennai

A vector store can be useful because it can
retrieve semantically related descriptions.

This is where vector retrieval is powerful.

# CELL 41

# Part 7 — A Comparison Table

| Requirement | Relational DB | Knowledge Graph | Vector Store |
|-------------|----------------|-----------------|--------------|
| Exact filtering | Excellent | Good | Weak |
| Transactions | Excellent | Usually not primary | Not primary |
| Aggregation | Excellent | Possible | Weak |
| Explicit relationships | Good | Excellent | Weak |
| Multi-hop traversal | Possible | Excellent | Weak |
| Semantic similarity | Weak | Possible | Excellent |
| Keyword-independent retrieval | Limited | Limited | Excellent |
| Explainable relationship path | Possible | Excellent | Limited |
| Structured numerical conditions | Excellent | Good | Weak |
| RAG retrieval | Possible | Possible | Excellent |

This table is not saying that a system cannot perform
a particular operation.

It indicates what each representation is naturally
good at.

# CELL 41A

# Hands-On Comparison

We have now implemented:

    Relational Database → SQLite

    Knowledge Graph → Python graph

    Vector Store → Ollama + Chroma

Now we will compare them experimentally.

We will use the SAME travel problem
and examine which representation is most natural
for different questions.

# CELL 41B

## Question 1

> Which hotels cost less than ₹3500?

Think before looking at the answer.

Which representation is the most natural?

    A. Relational Database
    B. Knowledge Graph
    C. Vector Store

# CELL 41C

### Observation

Relational Database:

    WHERE price < 3500

This is a direct structured condition.

Knowledge Graph:

    Hotel
       ↓ PRICE_PER_NIGHT
    Price

It can also answer the question.

Vector Store:

    Similarity search

It may retrieve relevant hotel documents,
but semantic similarity is not an exact
numerical filter.

### Conclusion

For this question:

> Relational Database is the most natural fit.

# CELL 41D

## Question 2

> Which destinations are near Chennai?

Think:

    A. Relational Database
    B. Knowledge Graph
    C. Vector Store

Which representation expresses the relationship
most directly?

# CELL 41E

### Observation

Knowledge Graph:

    Destination
         ↓ NEAR
       Chennai

The relationship is explicitly represented.

Relational Database:

    A separate relationship table
    and possibly JOIN operations are required.

Vector Store:

    Relevant text may be retrieved,
    but the NEAR relationship is not
    explicitly represented as a graph edge.

### Conclusion

> Knowledge Graph is the most natural fit
> for explicit relationship questions.

# CELL 41F

## Question 3

> I want a peaceful historical place near Chennai
> for a weekend trip.

Which representation is most natural?

    A. Relational Database
    B. Knowledge Graph
    C. Vector Store

# CELL 41G

### Observation

This question is expressed in terms of meaning:

    peaceful
    historical
    weekend
    place
    near Chennai

The user may express the same intention
using many different words.

A vector store can retrieve text based
on semantic similarity.

### Conclusion

> Vector Store is the most natural fit
> for semantic retrieval.

# CELL 41H

# What Did Our Experiments Show?

| Question Type | Natural Representation |
|---|---|
| Exact numerical filtering | Relational DB |
| Explicit relationships | Knowledge Graph |
| Multi-hop traversal | Knowledge Graph |
| Semantic similarity | Vector Store |
| Aggregation | Relational DB |
| Relevant text retrieval | Vector Store |

The important word is:

> NATURAL

The other systems may sometimes be able
to answer the question.

But one representation may make the problem
much more natural to express and solve.

# CELL 42

# Part 8 — Representation vs Query Type

This is the most important conceptual point
of this notebook.

Think about the QUESTION first.

### Question Type 1

    "Which hotels cost less than ₹3500?"

Best natural fit:

    Relational Database

because this is structured filtering.

### Question Type 2

    "Which heritage destinations near Chennai
     have hotels under ₹3500?"

Natural fit:

    Knowledge Graph

because several relationships are involved.

### Question Type 3

    "Find a peaceful historical place near Chennai."

Natural fit:

    Vector Store

because this is primarily a semantic retrieval problem.

Therefore:

    QUESTION
       ↓
    TYPE OF KNOWLEDGE
       ↓
    APPROPRIATE REPRESENTATION

# CELL 43

# Part 9 — The Important Correction

Do NOT conclude:

    Knowledge Graph > Relational Database > Vector Store

or:

    Vector Store > Knowledge Graph

There is no universal ranking.

Each solves a different class of problems well.

A better way to think is:

    Structured data
         → Relational DB

    Explicit relationships
         → Knowledge Graph

    Semantic similarity
         → Vector Store

In real systems, these technologies can also
be used TOGETHER.

# CELL 43A

# Integrated Travel Question

Consider the following user request:

> I want a peaceful heritage destination near Chennai
> with a hotel costing less than ₹3500 per night.

This question contains several types of knowledge.

Identify them before choosing a technology.

### Part 1

"heritage destination near Chennai"

What type of knowledge is this?

### Part 2

"hotel costing less than ₹3500"

What type of knowledge is this?

### Part 3

"peaceful"

What type of knowledge is this?

# CELL 43B

## Decomposing the Question

### Relationship knowledge

    heritage
    near Chennai

→ Knowledge Graph

### Structured numerical knowledge

    price < 3500

→ Relational Database

### Semantic preference

    peaceful

→ Vector Store

Therefore, the complete question is actually
a combination of different types of knowledge.

# CELL 43C

# A Hybrid Travel Assistant

A realistic system could combine all three.

                USER QUESTION
                      ↓
               QUESTION ANALYSIS
                      ↓
       ┌──────────────┼──────────────┐
       ↓              ↓              ↓
   Knowledge       Relational      Vector
     Graph           DB            Store
       ↓              ↓              ↓
 relationships      price        semantic
  and paths        filtering     preference
       └──────────────┼──────────────┘
                      ↓
                  Combined
                   Results
                      ↓
                     LLM
                      ↓
               Travel Answer

### Key idea

> Different representations can work together.

# CELL 44

# Part 10 — Hybrid Systems

A real Travel Planning application could use
all three.

For example:

    Relational Database
           ↓
    Hotel availability
    prices
    bookings
           ↓
    Knowledge Graph
           ↓
    destinations
    relationships
    attractions
           ↓
    Vector Store
           ↓
    travel descriptions
    reviews
    semantic preferences

An AI assistant could combine all three sources.

# CELL 45

## Example of a Hybrid Travel Query

User asks:

> I want a heritage destination near Chennai,
> with an affordable hotel, and I prefer peaceful
> places suitable for a weekend trip.

Different parts of the question may be handled
by different representations.

### Knowledge Graph

Find:

    heritage destination
    near Chennai

### Relational Database

Find:

    hotels
    price within budget
    availability

### Vector Store

Find descriptions matching:

    peaceful
    weekend
    relaxing

The final AI system can combine these results.

# CELL 46

# Part 11 — The Connection to RAG

This distinction is especially important
when building RAG systems.

A vector store is excellent for:

    "Find documents relevant to this question."

But suppose the question is:

> Which heritage destinations near Chennai
> have hotels below ₹3500?

A vector search alone may retrieve relevant text,
but it does not inherently guarantee exact
relationship traversal and structured filtering.

A better system might use:

    Vector Retrieval
          +
    Knowledge Graph
          +
    Structured Database

This is sometimes called a hybrid approach.

The important design principle is:

> Use each representation for the type of
> knowledge it handles best.

# CELL 47

# Part 12 — A Thinking Exercise

Suppose the travel application receives
the following questions.

For each question, decide which representation
is the most natural fit.

### Question A

Which hotels cost less than ₹3000?

### Question B

Which places are connected to Chennai
through two or more cities?

### Question C

Find a relaxing historical destination
for a weekend trip.

### Question D

How many hotels are available in each city?

### Question E

Which destination is near Chennai,
is heritage-listed, and has an affordable hotel?

Do not immediately think about technology.

First classify the QUESTION.

# CELL 48

# Suggested Thinking

### A

    Structured numerical filtering

→ Relational Database

### B

    Multi-hop relationship traversal

→ Knowledge Graph

### C

    Semantic retrieval

→ Vector Store

### D

    Aggregation

→ Relational Database

### E

    Multiple explicit relationships

→ Knowledge Graph

Notice that the same application
can require different representations.

# CELL 49

# Part 13 — The Central Idea of Knowledge Representation

We have now seen three ways of representing
the SAME travel domain.

### Relational

    TABLE
      ↓
    ROWS + COLUMNS

### Knowledge Graph

    NODE
      ↓
    RELATIONSHIP
      ↓
    NODE

### Vector

    TEXT
      ↓
    EMBEDDING
      ↓
    VECTOR

The representation determines what kinds
of operations become natural.

Therefore:

> Representation is not merely a storage decision.

It influences:

    how we query
    how we retrieve
    how we reason
    how we combine information

# CELL 50

# Part 14 — Blackboard Summary

The complete journey can be drawn as:

                 TRAVEL QUESTION
                        ↓
               What type of question?
                        ↓
        ┌───────────────┼────────────────┐
        ↓               ↓                ↓
   Structured       Relationship      Semantic
      Data             Data           Meaning
        ↓               ↓                ↓
 Relational DB     Knowledge Graph    Vector Store
        ↓               ↓                ↓
    SQL Query       Graph Query      Similarity
        ↓               ↓                ↓
    Exact Data       Traversal       Relevant Text

The important decision is:

> Choose the representation according
> to the nature of the question.

# CELL 51

# Part 15 — Why We Started with Python

So far we have deliberately avoided
specialized Knowledge Graph software.

Why?

Because students first need to understand
the concepts:

    Node
    Edge
    Triple
    Query
    Traversal
    Multi-Hop
    Representation

Now we have compared these concepts with
relational tables and vector representations.

Therefore, students are now ready to see
a real graph database.

The next step is:

    Neo4j
       +
    Cypher

# CELL 52

# Part 16 — Preparing for Neo4j

In the next notebook, we will move from:

    Python in-memory graph

to:

    Neo4j Graph Database

We will use the SAME Travel Planning problem.

Students will see:

Python representation:

    ("Chennai", "CONNECTED_TO", "Bengaluru")

become conceptually:

    (Chennai)-[:CONNECTED_TO]->(Bengaluru)

Then we will use Cypher to ask questions
such as:

    Which places are near Chennai?

    Which hotels are in heritage destinations?

    Is there a path from Chennai to Mysuru?

The important point is:

> Neo4j is not introducing the concept of a
> Knowledge Graph.

The concept was already learned.

Neo4j is providing a real platform for
storing and querying that graph.

# CELL 53

# Final Takeaways

### 1.

A relational database represents structured
information using tables.

### 2.

A Knowledge Graph represents knowledge using
entities and explicit relationships.

### 3.

A vector store represents semantic meaning
using embeddings.

### 4.

Relational databases are strong at:

    exact filtering
    aggregation
    transactions
    structured queries

### 5.

Knowledge Graphs are strong at:

    explicit relationships
    graph traversal
    multi-hop reasoning
    explainable paths

### 6.

Vector stores are strong at:

    semantic search
    similarity
    finding relevant information

### 7.

No representation is universally superior.

The key principle is:

# Choose the representation according to the problem.

# CELL 54

# Notebook 5 Complete

## The Journey So Far

KG-01

    Why do we need a Knowledge Graph?

        ↓

KG-02

    How do we build a Knowledge Graph?

        ↓

KG-03

    How do we query, traverse and reason?

        ↓

KG-04

    How do we build a KG from CSV data?

        ↓

KG-05

    When should we use:

        Relational Database
        Knowledge Graph
        Vector Store?

        ↓

KG-06

    How do we implement a real Knowledge Graph
    using Neo4j and Cypher?

The next notebook moves from
CONCEPTUAL GRAPH
to
REAL GRAPH DATABASE.

Do not choose the technology first; understand the question first.”

The students have now seen the same travel problem through three lenses:


                    TRAVEL KNOWLEDGE
                           │
             ┌─────────────┼─────────────┐
             ↓             ↓             ↓
          TABLE          GRAPH         VECTOR
             ↓             ↓             ↓
           SQL          Cypher       Similarity
             ↓             ↓             ↓
       Structured      Relations      Semantic
         Query          + Paths       Search